# Validazione end-to-end — PINN qMRI (rilassometria T2)

Walkthrough eseguibile: generazione dati sintetici, training della PINN non
supervisionata, baseline least-squares e confronto delle mappe T2.

Modello fisico: $S(TE) = S_0 \, e^{-TE/T_2}$. La PINN minimizza la coerenza con
questo modello, senza vedere la ground truth.

In [ ]:
import sys
from pathlib import Path

# Rendi importabile il package src/ quando si esegue il notebook.
SRC = Path.cwd().parents[0] / "src" if (Path.cwd().name == "notebooks") else Path.cwd() / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import matplotlib.pyplot as plt

from pinn_qmri.synth import generate
from pinn_qmri.train import TrainConfig, train_pinn
from pinn_qmri.baseline import fit_image
from pinn_qmri.metrics import all_metrics
from pinn_qmri.model import get_device

print("device:", get_device())

## 1. Dati sintetici con ground truth

In [ ]:
data = generate(height=32, width=32, num_echoes=8, noise_sigma=3.0, seed=0)
print("segnali:", data.signals.shape, "| TE (ms):", data.te)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(data.signals[:, :, i * 2], cmap="gray")
    ax.set_title(f"TE={data.te[i*2]:.0f} ms")
    ax.axis("off")
plt.show()

## 2. Training PINN (non supervisionato)

In [ ]:
cfg = TrainConfig(epochs=300, seed=0)
_, _, t2_pinn, history = train_pinn(data.signals, data.te, cfg)
print(f"loss iniziale {history[0]:.3e} -> finale {history[-1]:.3e}")

plt.figure(figsize=(5, 3))
plt.semilogy(history)
plt.xlabel("epoca"); plt.ylabel("physics loss"); plt.title("Convergenza PINN")
plt.grid(True, alpha=0.3); plt.show()

## 3. Baseline least-squares e confronto

In [ ]:
_, t2_ls = fit_image(data.te, data.signals)
m_pinn = all_metrics(t2_pinn, data.t2_true)
m_ls = all_metrics(t2_ls, data.t2_true)
print("PINN T2:", {k: round(v, 2) for k, v in m_pinn.items()})
print("LS   T2:", {k: round(v, 2) for k, v in m_ls.items()})

vmax = float(np.percentile(data.t2_true, 99))
fig, ax = plt.subplots(1, 3, figsize=(12, 3.6))
for a, img, t in zip(ax, [data.t2_true, t2_pinn, t2_ls], ["GT", "PINN", "LS"]):
    im = a.imshow(img, cmap="viridis", vmin=0, vmax=vmax); a.set_title(t); a.axis("off")
    fig.colorbar(im, ax=a, fraction=0.046, pad=0.04)
plt.show()

## 4. Conclusioni

La PINN, addestrata senza ground truth ma vincolata dalla fisica del decadimento
T2, ricostruisce la mappa T2 con errore confrontabile al fitting least-squares,
processando tutti i pixel in un singolo batch. Vedi `scripts/evaluate.py` per il
confronto su piu' livelli di rumore.